In [10]:
import nltk

def ngrams(sentenc, n):
    words = sentence.split()
    ngrams = zip(*[words[i:] for i in range(n)])
    return list(ngrams)
sentence = '안녕하세요. 만나서 진심으로 반가워요.'

unigram = ngrams(sentence, 1)
bigram = ngrams(sentence, 2)
trigram = ngrams(sentence, 2)

print(unigram)
print(bigram)
print(trigram)

unigram = nltk.ngrams(sentence.split(), 1)
bigram = nltk.ngrams(sentence.split(), 2)
trigram = nltk.ngrams(sentence.split(), 3)

print(list(unigram))
print(list(bigram))
print(list(trigram))

[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요.',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요.')]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요.')]
[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요.',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요.')]
[('안녕하세요.', '만나서', '진심으로'), ('만나서', '진심으로', '반가워요.')]


In [11]:
# 벡터화

from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "That movie is famous movie",
    "I like that actor",
    "I don't like that actor"
]

tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer.fit(corpus)
tfidf_matrix = tfidf_vectorizer.transform(corpus)

print(tfidf_matrix.toarray())
print(tfidf_vectorizer.vocabulary_)

[[0.         0.         0.39687454 0.39687454 0.         0.79374908
  0.2344005 ]
 [0.61980538 0.         0.         0.         0.61980538 0.
  0.48133417]
 [0.4804584  0.63174505 0.         0.         0.4804584  0.
  0.37311881]]
{'that': 6, 'movie': 5, 'is': 3, 'famous': 2, 'like': 4, 'actor': 0, 'don': 1}


In [12]:
import torch.nn as nn

class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = embedding_dim
        )
        self.linear = nn.Linear(
            in_features = embedding_dim,
            out_features = vocab_size
        )

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output = self.linear(embeddings)
        return output

In [13]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load('nsmc')
corpus = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\use\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\use\K

In [14]:
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])

[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


In [15]:
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus = tokens, n_vocab = 5000, special_tokens = ['<unk>'])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [16]:
def get_word_pairs(tokens, window_size):
    pairs = []

    for sentence in tokens:
        sentence_length = len(sentence)

        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx - window_size)
            window_end = min(sentence_length, idx + window_size + 1)

            context_words = (
                sentence[window_start:idx]
                + sentence[idx + 1:window_end]
            )

            for context_word in context_words:
                pairs.append([center_word, context_word])

    return pairs


word_pairs = get_word_pairs(tokens, window_size=2)
print(len(word_pairs))
print(word_pairs[:5])

2589946
[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [17]:
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id["<unk>"]
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs


index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])
print(len(vocab))

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]
5001


In [18]:
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs_tensor = torch.tensor(index_pairs)
center_indexes = index_pairs_tensor[:, 0]
context_indexes = index_pairs_tensor[:, 1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)


In [19]:
import torch.optim as optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

word2vec = VanillaSkipgram(vocab_size = len(token_to_id), embedding_dim = 128).to(device)

criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr = 0.1)


cuda


In [20]:
for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss
    cost = cost / len(dataloader)
    print(epoch + 1, cost)


1 tensor(6.1947, device='cuda:0', grad_fn=<DivBackward0>)
2 tensor(5.9817, device='cuda:0', grad_fn=<DivBackward0>)
3 tensor(5.9324, device='cuda:0', grad_fn=<DivBackward0>)
4 tensor(5.9025, device='cuda:0', grad_fn=<DivBackward0>)
5 tensor(5.8804, device='cuda:0', grad_fn=<DivBackward0>)
6 tensor(5.8627, device='cuda:0', grad_fn=<DivBackward0>)
7 tensor(5.8480, device='cuda:0', grad_fn=<DivBackward0>)
8 tensor(5.8351, device='cuda:0', grad_fn=<DivBackward0>)
9 tensor(5.8236, device='cuda:0', grad_fn=<DivBackward0>)
10 tensor(5.8131, device='cuda:0', grad_fn=<DivBackward0>)


In [21]:
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding

index = 30
token = vocab[index]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

연기
[ 2.3012543   0.36090037 -0.23686613  0.84599656  0.46788615  1.2151183
  0.20133843 -0.33450413  0.6225707   0.9551261  -1.2414246   0.85254216
 -0.14296569 -0.50892276  0.6919227   0.23368585 -0.23468946  2.1929567
 -0.04271371 -1.1583925   0.19183128 -0.25588322 -0.6247781  -1.6420565
  1.31527    -0.23809919 -1.1230423   0.48037115 -0.06875256 -0.6024129
  1.3023818   0.08127956  1.3797821   1.1641399   0.9467166   1.7947862
 -0.48419535 -1.0464448   0.5364805  -1.642609   -0.51944155  0.09675378
 -0.24958159  0.83751345  0.27779028 -0.23172283  0.47656167 -0.4283983
  0.574388    0.21182552  0.6653725   0.9639473  -0.27190548 -0.40157655
 -1.1593299   1.293587    0.1945366  -0.5581262  -0.23486513  0.49518487
  0.42826772  0.22497262 -0.15989861  1.7182262   1.3043234  -1.9171484
 -0.67206043  0.4262178   0.21428859 -0.2119561  -0.33455408  0.16742429
  0.2644703  -0.85129553  2.3597586   2.66023    -1.3335565  -0.04863327
 -0.3796385  -0.11633738 -0.14266096 -1.2281203   0.527

In [22]:
import numpy as np
from numpy.linalg import norm


def cosine_similarity(a, b):
    cosine = np.dot(b, a) / (norm(b, axis=1) * norm(a))
    return cosine

def top_n_index(cosine_matrix, n):
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1 : n + 1]
    return top_n


cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n=5)

print(f"{token}와 가장 유사한 5 개 단어")
for index in top_n:
    print(f"{id_to_token[index]} - 유사도 : {cosine_matrix[index]:.4f}")

연기와 가장 유사한 5 개 단어
공포물 - 유사도 : 0.3379
짜 - 유사도 : 0.2891
역작 - 유사도 : 0.2834
와서 - 유사도 : 0.2796
실제 - 유사도 : 0.2778
